PREPROCESSING PIPELINE OBJECT

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  FASE 10 — Cell 1: PREPROCESSING PIPELINE OBJECT            ║
# ║  Export pipeline yang bisa dipanggil inference.py            ║
# ╚══════════════════════════════════════════════════════════════╝

import sys, os, re, logging
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# ── Path Setup ─────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import (
    DATA_RAW_DIR, DATA_PROCESSED_DIR, MODELS_ML_DIR,
    MODELS_FINAL_DIR, GLOBAL_SEED, LABEL_MAP,
    W_WARNING_HRS, W_CRITICAL_HRS,
)

# ── Logging ────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="  %(levelname)s | %(message)s",
)
log = logging.getLogger("fase10.pipeline")

# ── Konstanta Lokal ─────────────────────────────────────────────
SEQ_NUMERIC_COLS = [
    "temperature", "vibration", "pressure",
    "rpm", "power_consumption", "noise_level",
]
ALL_SENSOR_COLS = SEQ_NUMERIC_COLS + ["humidity", "operating_hours"]
ROLL_WINDOWS    = [24, 48]
LAG_SIZES       = [1, 6, 24]

# Mapping kategori NLP (hardcoded — JANGAN diganti LabelEncoder baru)
DAMAGE_CAT_MAP  = {"electrical": 0, "lubrication": 1, "mechanical": 2, "unknown": 3}

# Kolom yang akan di-drop sebelum scaling
DROP_COLS = [
    "timestamp", "machine_id", "failure", "health_label",
    "health_label_confirmed", "health_label_encoded",
    "rul_days", "technician_notes", "damage_category_raw",
]

SEP = "=" * 65
print(SEP)
print("  FASE 10 — Cell 1: PREPROCESSING PIPELINE OBJECT")
print(SEP)


# ══════════════════════════════════════════════════════════════
# LANGKAH 2 — CUSTOM TRANSFORMER CLASS
# ══════════════════════════════════════════════════════════════

class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    """
    Stateless sklearn transformer yang mereproduksi seluruh
    feature engineering dari Fase 5 dan 6 secara deterministik.

    Input : DataFrame raw sensor_readings (n_samples x raw_cols)
    Output: numpy array (n_samples x 69) — siap masuk scaler
    """

    def __init__(self, maintenance_df=None):
        """
        Parameters
        ----------
        maintenance_df : pd.DataFrame atau None
            Tabel maintenance_logs opsional. Jika None,
            hours_since_last_maint akan di-set ke 0.
        """
        self.maintenance_df = maintenance_df

    def fit(self, X, y=None):
        """Stateless — tidak ada parameter yang di-fit."""
        # Ambil urutan kolom dari X_train_clf saat fit pertama kali
        self._expected_cols = self._load_expected_cols()
        return self

    # ── Helpers Internal ────────────────────────────────────
    @staticmethod
    def _load_expected_cols():
        """Load urutan 69 kolom dari X_train_clf.parquet."""
        train_path = DATA_PROCESSED_DIR / "X_train_clf.parquet"
        if train_path.exists():
            cols = pd.read_parquet(train_path).columns.tolist()
            log.info(f"Expected feature cols loaded: {len(cols)} kolom")
            return cols
        log.warning("X_train_clf.parquet tidak ditemukan — gunakan urutan default.")
        return None

    def _clip_negatives(self, df):
        """Clip vibration ke minimum 0 (fix DFT-02 dari Fase 5)."""
        if "vibration" in df.columns:
            df["vibration"] = df["vibration"].clip(lower=0)
        return df

    def _add_rolling_features(self, df):
        """
        Tambahkan rolling mean, std, max untuk 6 sensor utama
        pada window 24h dan 48h, dikelompokkan per machine_id.
        Menghasilkan 6 x 2 x 3 = 36 kolom baru.
        """
        df = df.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)
        for sensor in SEQ_NUMERIC_COLS:
            for w in ROLL_WINDOWS:
                grp = df.groupby("machine_id")[sensor]
                df[f"{sensor}_roll_mean_{w}h"] = grp.transform(
                    lambda s: s.rolling(w, min_periods=1).mean()
                )
                df[f"{sensor}_roll_std_{w}h"]  = grp.transform(
                    lambda s: s.rolling(w, min_periods=1).std()
                ).fillna(0)
                df[f"{sensor}_roll_max_{w}h"]  = grp.transform(
                    lambda s: s.rolling(w, min_periods=1).max()
                )
        log.info("Rolling features added: 36 kolom")
        return df

    def _add_lag_features(self, df):
        """
        Tambahkan lag features untuk 6 sensor utama pada lag 1h, 6h, 24h
        per machine_id. NaN lag diisi dengan nilai asli sensor.
        Menghasilkan 6 x 3 = 18 kolom baru.
        """
        for sensor in SEQ_NUMERIC_COLS:
            for lag in LAG_SIZES:
                col_name = f"{sensor}_lag_{lag}h"
                df[col_name] = (
                    df.groupby("machine_id")[sensor]
                    .shift(lag)
                )
                # Fill NaN dengan nilai asli (ffill lalu bfill sebagai fallback)
                df[col_name] = df[col_name].fillna(df[sensor])
        log.info("Lag features added: 18 kolom")
        return df

    def _add_ratio_features(self, df):
        """
        Tambahkan 4 cross-sensor interaction dan ratio features.
        """
        df["temp_vibration_ratio"]            = df["temperature"] / df["vibration"].clip(lower=0.001)
        df["pressure_rpm_ratio"]              = df["pressure"]    / df["rpm"].clip(lower=1)
        df["power_noise_ratio"]               = df["power_consumption"] / df["noise_level"].clip(lower=0.001)
        df["vibration_pressure_interaction"]  = df["vibration"]   * df["pressure"]
        log.info("Ratio/interaction features added: 4 kolom")
        return df

    def _add_degradation_proxy(self, df):
        """
        Tambahkan hours_since_last_maint per machine_id.
        Jika maintenance_df tersedia, hitung menggunakan merge_asof.
        Jika tidak, fill dengan 0 (safe default untuk real-time inference).
        """
        if self.maintenance_df is not None:
            maint = (
                self.maintenance_df[["machine_id", "timestamp"]]
                .drop_duplicates()
                .sort_values(["machine_id", "timestamp"])
                .rename(columns={"timestamp": "maint_timestamp"})
            )
            df = df.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)
            merged_parts = []
            for mid, grp in df.groupby("machine_id"):
                maint_mid = maint[maint["machine_id"] == mid]
                grp_sorted = grp.sort_values("timestamp")
                if len(maint_mid) > 0:
                    result = pd.merge_asof(
                        grp_sorted,
                        maint_mid.sort_values("maint_timestamp"),
                        left_on="timestamp", right_on="maint_timestamp",
                        by="machine_id",
                    )
                    result["hours_since_last_maint"] = (
                        (result["timestamp"] - result["maint_timestamp"])
                        .dt.total_seconds() / 3600
                    ).fillna(0).clip(lower=0)
                else:
                    grp_sorted["hours_since_last_maint"] = 0
                    result = grp_sorted
                merged_parts.append(result)
            df = pd.concat(merged_parts).sort_index().reset_index(drop=True)
            if "maint_timestamp" in df.columns:
                df = df.drop(columns=["maint_timestamp"])
        else:
            df["hours_since_last_maint"] = 0
        log.info("Degradation proxy added: 1 kolom (hours_since_last_maint)")
        return df

    def _add_nlp_features(self, df):
        """
        Ekstrak damage_category dan severity_score dari 'technician_notes'.
        Menggunakan keyword matching sederhana — tidak ada model NLP eksternal.
        Jika kolom tidak ada, gunakan nilai default (unknown / 1).
        """
        MECHANICAL_KW   = ["belt", "pulley", "rantai"]
        ELECTRICAL_KW   = ["sensor", "elektrik", "listrik", "kabel"]
        LUBRICATION_KW  = ["oli", "pelumas", "grease"]
        SEVERITY_HIGH   = ["putus", "rusak parah", "breakdown"]
        SEVERITY_MED    = ["aus", "bocor", "abnormal"]
        SEVERITY_LOW    = ["inspeksi", "rutin", "normal"]

        def classify_damage(note):
            if not isinstance(note, str):
                return "unknown"
            note_lower = note.lower()
            if any(kw in note_lower for kw in MECHANICAL_KW):
                return "mechanical"
            if any(kw in note_lower for kw in ELECTRICAL_KW):
                return "electrical"
            if any(kw in note_lower for kw in LUBRICATION_KW):
                return "lubrication"
            return "unknown"

        def score_severity(note):
            if not isinstance(note, str):
                return 1
            note_lower = note.lower()
            if any(kw in note_lower for kw in SEVERITY_HIGH):
                return 3
            if any(kw in note_lower for kw in SEVERITY_MED):
                return 2
            return 1

        if "technician_notes" in df.columns:
            df["damage_category_raw"] = df["technician_notes"].apply(classify_damage)
            df["severity_score"]      = df["technician_notes"].apply(score_severity)
        else:
            df["damage_category_raw"] = "unknown"
            df["severity_score"]      = 1

        # Encode dengan mapping tetap (tidak fit LabelEncoder baru)
        df["damage_category"] = df["damage_category_raw"].map(DAMAGE_CAT_MAP).fillna(3).astype(int)
        log.info("NLP features added: damage_category + severity_score")
        return df

    def _select_and_order_features(self, df):
        """
        Drop kolom non-fitur dan reindex ke urutan 69 kolom yang tepat.
        Kolom yang tidak ada di DataFrame akan di-fill dengan 0.
        """
        df = df.drop(columns=DROP_COLS, errors="ignore")

        expected = getattr(self, "_expected_cols", None)
        if expected is None:
            expected = self._load_expected_cols()

        if expected is not None:
            # Reindex — kolom hilang diisi 0, kolom ekstra dibuang
            df = df.reindex(columns=expected, fill_value=0)
        else:
            log.warning("Expected cols tidak ditemukan — output mungkin tidak 69 kolom!")

        return df

    def transform(self, X, y=None):
        """
        Jalankan pipeline feature engineering dari awal sampai akhir.

        Parameters
        ----------
        X : pd.DataFrame
            DataFrame mentah (format sensor_readings.csv).
        y : diabaikan

        Returns
        -------
        numpy.ndarray  shape (n_samples, 69)
        """
        df = X.copy()
        log.info(f"transform() dipanggil | Input shape: {df.shape}")

        df = self._clip_negatives(df)
        df = self._add_rolling_features(df)
        df = self._add_lag_features(df)
        df = self._add_ratio_features(df)
        df = self._add_degradation_proxy(df)
        df = self._add_nlp_features(df)
        df = self._select_and_order_features(df)

        log.info(f"transform() selesai | Output shape: {df.shape}")
        return df


# ══════════════════════════════════════════════════════════════
# LANGKAH 3 — BANGUN PIPELINE OBJECT
# ══════════════════════════════════════════════════════════════
print("\n  [1/4] Building pipeline object...")

# Load scaler yang sudah di-fit pada Train Set (M-01 s/d M-14)
# JANGAN refit — cukup inject ke dalam Pipeline sebagai step ke-2
scaler_path = MODELS_ML_DIR / "scaler.pkl"
if not scaler_path.exists():
    raise FileNotFoundError(f"Scaler tidak ditemukan di: {scaler_path}")
scaler = joblib.load(scaler_path)
log.info(f"Scaler loaded dari: {scaler_path}")

# Buat instance transformer (dengan fit dulu untuk load expected_cols)
feat_transformer = FeatureEngineeringTransformer(maintenance_df=None)
feat_transformer._expected_cols = FeatureEngineeringTransformer._load_expected_cols()

# Assembling sklearn Pipeline
preprocessing_pipeline = Pipeline(steps=[
    ("feature_engineering", feat_transformer),
    ("scaling",             scaler),          # Scaler sudah di-fit, TIDAK di-refit ulang
])

print("  Pipeline steps:")
for step_name, step_obj in preprocessing_pipeline.steps:
    print(f"    [{step_name}] → {type(step_obj).__name__}")


# ══════════════════════════════════════════════════════════════
# LANGKAH 4 — SMOKE TEST
# ══════════════════════════════════════════════════════════════
print("\n  [2/4] Running smoke test...")

raw_df = pd.read_csv(DATA_RAW_DIR / "sensor_readings.csv", nrows=100)

# Pastikan kolom timestamp terbaca sebagai datetime
raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"])

# Ambil 5 baris dari satu mesin agar rolling tidak terlalu pendek
sample_df = (
    raw_df[raw_df["machine_id"] == raw_df["machine_id"].iloc[0]]
    .head(50)    # ambil 50 dulu untuk rolling window yang sehat
)

smoke_output = preprocessing_pipeline.transform(sample_df)

# Ambil 5 baris terakhir (rolling sudah stabil)
sample_output = smoke_output[-5:]

print(f"\n  Output shape    : {sample_output.shape}    ← harus (5, 69)")
print(f"  Any NaN         : {np.isnan(sample_output).any()}   ← harus False")
print(f"  Output dtype    : {sample_output.dtype}  ← harus float64")
print(f"  Sample row[0]   : {sample_output[0, :3]}")

assert sample_output.shape[1] == 69, f"❌ Kolom bukan 69! Got {sample_output.shape[1]}"
assert not np.isnan(sample_output).any(), "❌ Ada NaN di output!"
assert sample_output.dtype == np.float64, f"❌ Dtype bukan float64! Got {sample_output.dtype}"
print("\n  ✅ Smoke test PASSED.")


# ══════════════════════════════════════════════════════════════
# LANGKAH 5 — EXPORT & VERIFIKASI
# ══════════════════════════════════════════════════════════════
print("\n  [3/4] Exporting pipeline...")

MODELS_FINAL_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_PATH = MODELS_FINAL_DIR / "preprocessing_pipeline.pkl"

joblib.dump(preprocessing_pipeline, PIPELINE_PATH)
pipeline_size_kb = PIPELINE_PATH.stat().st_size / 1024
log.info(f"Pipeline disimpan ke: {PIPELINE_PATH} ({pipeline_size_kb:.1f} KB)")

# ── Verifikasi post-reload ────────────────────────────────────
print("\n  [4/4] Verifying post-reload...")
pipeline_check = joblib.load(PIPELINE_PATH)
check_output   = pipeline_check.transform(sample_df)[-5:]

assert check_output.shape == (5, 69), \
    f"Shape mismatch setelah reload! Got {check_output.shape}"
assert not np.isnan(check_output).any(), "NaN terdeteksi setelah reload!"
print(f"  ✅ Pipeline verified post-reload. Shape: {check_output.shape}")

print(f"\n{SEP}")
print(f"  ✅ PREPROCESSING PIPELINE SELESAI DIEKSPOR")
print(f"  Path : {PIPELINE_PATH}")
print(f"  Size : {pipeline_size_kb:.1f} KB")
print(f"  Steps: feature_engineering → scaling (StandardScaler, pre-fitted)")
print(f"  Usage: features = preprocessing_pipeline.transform(raw_df)")
print(f"         → output shape (n_samples, 69), dtype float64")
print(SEP)
print("  ✅ Cell 1 selesai — lanjut ke Cell 2 (Model Artifacts Export).")


  INFO | Scaler loaded dari: C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\models\ml_track\scaler.pkl
  INFO | Expected feature cols loaded: 69 kolom
  INFO | transform() dipanggil | Input shape: (5, 11)
  INFO | Rolling features added: 36 kolom
  INFO | Lag features added: 18 kolom
  INFO | Ratio/interaction features added: 4 kolom
  INFO | Degradation proxy added: 1 kolom (hours_since_last_maint)
  INFO | NLP features added: damage_category + severity_score
  INFO | transform() selesai | Output shape: (5, 69)


  FASE 10 — Cell 1: PREPROCESSING PIPELINE OBJECT

  [1/4] Building pipeline object...
  Pipeline steps:
    [feature_engineering] → FeatureEngineeringTransformer
    [scaling] → StandardScaler

  [2/4] Running smoke test...


c:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\venv\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
  INFO | Pipeline disimpan ke: C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\models\final\preprocessing_pipeline.pkl (6.0 KB)
  INFO | transform() dipanggil | Input shape: (5, 11)
  INFO | Rolling features added: 36 kolom
  INFO | Lag features added: 18 kolom
  INFO | Ratio/interaction features added: 4 kolom
  INFO | Degradation proxy added: 1 kolom (hours_since_last_maint)
  INFO | NLP features added: damage_category + severity_score
  INFO | transform() selesai | Output shape: (5, 69)



  Output shape    : (5, 69)    ← harus (5, 69)
  Any NaN         : False   ← harus False
  Output dtype    : float64  ← harus float64
  Sample row[0]   : [-0.14141558  0.23886368  0.92789419]

  ✅ Smoke test PASSED.

  [3/4] Exporting pipeline...

  [4/4] Verifying post-reload...
  ✅ Pipeline verified post-reload. Shape: (5, 69)

  ✅ PREPROCESSING PIPELINE SELESAI DIEKSPOR
  Path : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\models\final\preprocessing_pipeline.pkl
  Size : 6.0 KB
  Steps: feature_engineering → scaling (StandardScaler, pre-fitted)
  Usage: features = preprocessing_pipeline.transform(raw_df)
         → output shape (n_samples, 69), dtype float64
  ✅ Cell 1 selesai — lanjut ke Cell 2 (Model Artifacts Export).


c:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\venv\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
